In [3]:
#!/usr/bin/env python3

from pathlib import Path
import re
import shutil
import numpy as np
from scipy.io import wavfile
import os
import stat
import time

# =========================================================
# SETTINGS
# =========================================================
ROOT = Path(".").resolve()

TEST_DIR = ROOT / "test"
TEST_SPLICED_DIR = TEST_DIR / "spliced"
TEST_READY_DIR = ROOT / "test_ready"

TARGET_LABELS = ["får", "ged", "hest", "laks", "ulv"]
OPTIONAL_UNKNOWN_LABELS = ["stilhed", "unknown"]

WINDOW_S = 1.00
PRE_S = 0.20
MIN_GAP_S = 0.45
FRAME_MS = 20
HOP_MS = 10
THRESH_MULT = 6.0
MAX_SLICES_PER_FILE = 10

# Hvis enkelte testfiler skal tunes særskilt, kan de sættes her
SPECIAL_FILE_PARAMS = {
    # "test_får_C_ukendt_tmlKlasse.001.wav": {
    #     "window_s": 1.0,
    #     "pre_s": 0.22,
    #     "min_gap_s": 0.55,
    #     "frame_ms": 20,
    #     "hop_ms": 10,
    #     "thresh_mult": 4.5,
    #     "max_slices": 2,
    # },
}

# Matcher fx:
# test_får_C_ukendt_tmlKlasse.001.wav
# test_stilhed_ukendt_tmlKlasse.001.wav  (uden speaker)
TEST_FNAME_RE = re.compile(
    r"^test_(?P<label>[^_]+)"
    r"(?:_(?P<speaker>[A-Za-z]))?"
    r"_(?P<dist>[^_]+)"
    r"_(?P<env>[^.]+)"
    r"\.(?P<take>\d+)\.wav$",
    re.IGNORECASE
)

# Matcher splicede testfiler:
# test_får_C_ukendt_tmlKlasse_001_s01.wav
TEST_SPLICED_RE = re.compile(
    r"^test_(?P<label>[^_]+)"
    r"(?:_(?P<speaker>[A-Za-z]))?"
    r"_(?P<dist>[^_]+)"
    r"_(?P<env>[^_]+)"
    r"_(?P<take>\d+)_s(?P<slice>\d+)\.wav$",
    re.IGNORECASE
)

# =========================================================
# HELPERS
# =========================================================
def sanitize(s: str) -> str:
    s = s.strip()
    return re.sub(r'[\\/:*?"<>|]+', "_", s)

def to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x
    return x.mean(axis=1)

def _handle_remove_readonly(func, path, exc):
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception as e:
        print(f"[cleanup] Could not delete {path}: {e}")

def safe_rmtree(path: Path, retries: int = 5, delay: float = 0.5):
    if not path.exists():
        return

    for attempt in range(1, retries + 1):
        try:
            shutil.rmtree(path, onerror=_handle_remove_readonly)
            return
        except PermissionError as e:
            print(f"[cleanup] PermissionError deleting {path} (attempt {attempt}/{retries}): {e}")
            time.sleep(delay)

    raise PermissionError(f"Could not delete folder after {retries} attempts: {path}")

def ensure_clean_dir(path: Path):
    if path.exists():
        safe_rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def moving_rms(x: np.ndarray, frame: int, hop: int) -> np.ndarray:
    if len(x) < frame:
        return np.array([], dtype=np.float32)
    n = 1 + (len(x) - frame) // hop
    rms = np.empty(n, dtype=np.float32)
    for i in range(n):
        seg = x[i * hop : i * hop + frame]
        rms[i] = np.sqrt(np.mean(seg * seg) + 1e-12)
    return rms

def detect_event_starts(rms: np.ndarray, thr: float, hop: int) -> np.ndarray:
    if rms.size == 0:
        return np.array([], dtype=np.int64)
    above = rms > thr
    starts = np.where(np.logical_and(above, np.concatenate([[False], ~above[:-1]])))[0]
    return (starts * hop).astype(np.int64)

def parse_test_base_name(wav_path: Path) -> str:
    m = TEST_FNAME_RE.match(wav_path.name)
    if m:
        label = sanitize(m.group("label"))
        speaker = m.group("speaker")
        dist = sanitize(m.group("dist"))
        env = sanitize(m.group("env"))
        take = int(m.group("take"))

        if speaker:
            speaker = sanitize(speaker)
            return f"test_{label}_{speaker}_{dist}_{env}_{take:03d}"
        else:
            return f"test_{label}_{dist}_{env}_{take:03d}"

    return sanitize(wav_path.stem)

# =========================================================
# STEP 1: SPLICE TEST FILES
# =========================================================
def slice_test_file(wav_path: Path, out_dir: Path, cfg: dict) -> int:
    fs, data = wavfile.read(wav_path)
    x = to_mono(data).astype(np.float32)

    frame = int(fs * (cfg["frame_ms"] / 1000.0))
    hop = int(fs * (cfg["hop_ms"] / 1000.0))

    rms = moving_rms(x, frame, hop)
    if rms.size == 0:
        return 0

    med = float(np.median(rms))
    thr = max(med * cfg["thresh_mult"], 1e-6)

    starts = detect_event_starts(rms, thr, hop)

    min_gap = int(fs * cfg["min_gap_s"])
    filtered = []
    last = -10**18
    for s in starts:
        if s - last >= min_gap:
            filtered.append(s)
            last = s
        if len(filtered) >= cfg["max_slices"]:
            break

    if not filtered:
        return 0

    base = parse_test_base_name(wav_path)
    out_dir.mkdir(parents=True, exist_ok=True)

    win = int(fs * cfg["window_s"])
    pre = int(fs * cfg["pre_s"])

    written = 0
    for i, s in enumerate(filtered, start=1):
        start = max(int(s - pre), 0)
        end = start + win
        if end > len(x):
            continue

        clip = x[start:end]
        clip = np.clip(clip, -32768, 32767).astype(np.int16)

        out_path = out_dir / f"{base}_s{i:02d}.wav"
        wavfile.write(out_path, fs, clip)
        written += 1

    return written

def slice_test_file_no_threshold(wav_path: Path, out_dir: Path) -> int:
    fs, data = wavfile.read(wav_path)
    x = to_mono(data).astype(np.int16)

    samples_per_chunk = int(fs * WINDOW_S)
    n_chunks = len(x) // samples_per_chunk

    base = parse_test_base_name(wav_path)
    out_dir.mkdir(parents=True, exist_ok=True)

    written = 0
    for i in range(n_chunks):
        start = i * samples_per_chunk
        end = start + samples_per_chunk
        clip = x[start:end]

        out_path = out_dir / f"{base}_s{i+1:02d}.wav"
        wavfile.write(out_path, fs, clip)
        written += 1

    return written

def splice_test():
    ensure_clean_dir(TEST_SPLICED_DIR)

    wavs = sorted([p for p in TEST_DIR.glob("*.wav") if p.is_file()])
    print(f"[test] Found {len(wavs)} raw test files")

    total = 0
    default_cfg = {
        "window_s": WINDOW_S,
        "pre_s": PRE_S,
        "min_gap_s": MIN_GAP_S,
        "frame_ms": FRAME_MS,
        "hop_ms": HOP_MS,
        "thresh_mult": THRESH_MULT,
        "max_slices": MAX_SLICES_PER_FILE,
    }

    for wav in wavs:
        cfg = SPECIAL_FILE_PARAMS.get(wav.name, default_cfg)

        # Hvis filen er stilhed, så slice uden threshold
        if wav.name.lower().startswith("test_stilhed_"):
            n = slice_test_file_no_threshold(wav, TEST_SPLICED_DIR)
            print(f"[test][no-threshold] {wav.name}: {n} slices")
        else:
            n = slice_test_file(wav, TEST_SPLICED_DIR, cfg)
            print(f"[test] {wav.name}: {n} slices")

        total += n

    print(f"[test] Total slices: {total}")

# =========================================================
# STEP 2: ORGANIZE TEST SLICES BY LABEL
# =========================================================
def build_test_ready():
    ensure_clean_dir(TEST_READY_DIR)

    labels_to_make = TARGET_LABELS + ["unknown"]
    for label in labels_to_make:
        (TEST_READY_DIR / label).mkdir(parents=True, exist_ok=True)

    wavs = sorted([p for p in TEST_SPLICED_DIR.glob("*.wav") if p.is_file()])
    moved = 0

    for wav in wavs:
        m = TEST_SPLICED_RE.match(wav.name)
        if not m:
            print(f"[test_ready] Could not parse filename: {wav.name}")
            continue

        label = m.group("label")

        # fold stilhed ind i unknown
        if label == "stilhed":
            label = "unknown"

        # alt ukendt kan også foldes ind i unknown
        if label not in TARGET_LABELS:
            label = "unknown"
 
        shutil.copy2(wav, TEST_READY_DIR / label / wav.name)
        moved += 1

    print(f"[test_ready] Copied {moved} files")
    for label in labels_to_make:
        count = len(list((TEST_READY_DIR / label).glob("*.wav")))
        print(f"[test_ready] {label}: {count} files")

# =========================================================
# MAIN
# =========================================================
def main():
    splice_test()
    build_test_ready()
    print("\nTest pipeline complete.")

if __name__ == "__main__":
    main()

[test] Found 34 raw test files
[test] test_får_A_kortAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_får_A_langAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_får_C_kortAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_får_C_langAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_får_J_kortAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_får_J_langAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_ged_A_kortAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_ged_A_langAfstand_stilleTmlKlasse.001.wav: 1 slices
[test] test_ged_C_kortAfstand_stilleTmlKlasse.001.wav: 9 slices
[test] test_ged_C_langAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_ged_J_kortAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_ged_J_langAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_hest_A_kortAfstand_stilleTmlKlasse.001.wav: 10 slices
[test] test_hest_A_langAfstand_stilleTmlKlasse.001.wav: 6 slices
[test] test_hest_C_kortAfstand_stilleTmlKlasse.001.wav: 10 s